In [0]:
%run ../config/pipeline_config

In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_raw_to_bronze(dataset_key):
    """
    Modular function to ingest raw CSV data into the Bronze Delta layer 
    using Unity Catalog compatible metadata[cite: 51, 68].
    """
    table_info = TABLES[dataset_key]
    raw_source_path = f"{PATHS['raw']}/{table_info['raw_file']}"
    target_table = f"{SCHEMA_NAME}.{table_info['bronze']}"
    
    print(f"Starting ingestion for: {dataset_key}")

    # Rule 1: Read CSV with Zero Transformations
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(raw_source_path)

    # Rule 2: Append Mandatory Metadata 
    # We use '_metadata.file_path' to be compliant with Unity Catalog
    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp()) \
                      .withColumn("source_file_name", col("_metadata.file_path"))

    # Write to Bronze layer as a Delta Table
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    
    print(f"✅ Successfully created table: {target_table}")

# Execute ingestion for all mandatory datasets
for key in TABLES.keys():
    ingest_raw_to_bronze(key)

In [0]:
display(spark.sql(f"SELECT source_file_name, ingestion_timestamp, * FROM {SCHEMA_NAME}.bronze_customers LIMIT 5"))